# Mamba vs. Transformer — Porównanie zdolności klasyfikacji na zbiorze danych IMDb

## 1. Setup i biblioteki

Możemy tę sekcję pominąć i wykonać komendę `uv sync`, jeśli notebook uruchamiamy lokalnie.

### 1.1 Instalacja bibliotek

#### 1.1.1 Konkretna wersja `pytorch`

In [ ]:
%pip install --force-reinstall torch==2.5.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

#### 1.1.2 Pobranie skompilowanej Mamby


In [ ]:
import sys
import torch

py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])

print(f"Detected Python: {py_ver}")
print(f"Detected PyTorch: torch{torch_ver}")

base_conv = f"causal_conv1d-1.6.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"
base_mamba = f"mamba_ssm-2.3.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"

conv_url = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/{base_conv}"
mamba_url = f"https://github.com/state-spaces/mamba/releases/download/v2.3.0/{base_mamba}"

%pip install packaging ninja --quiet

print("\nUnpacking Causal-Conv1d...")
%pip install {conv_url} --no-build-isolation

print("\nUnpacking Mamba-SSM...")
%pip install {mamba_url} --no-build-isolation


#### 1.1.3 Reszta bibliotek

In [ ]:
%pip install transformers datasets accelerate evaluate scikit-learn

## 1.2 Monkeypatch Mamby

In [3]:
# Targeted diagnostic
import pkgutil
import mamba_ssm.ops.triton as triton_pkg
import causal_conv1d

print("=== mamba_ssm.ops.triton submodules ===")
print([m.name for m in pkgutil.iter_modules(triton_pkg.__path__)])

try:
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
    print("\nselective_state_update: FOUND in triton.selective_state_update")
except Exception as e:
    print(f"\nselective_state_update: NOT FOUND — {e}")

print("\n=== causal_conv1d top-level ===")
print([x for x in dir(causal_conv1d) if not x.startswith("_")])

try:
    from causal_conv1d import causal_conv1d_update
    print("\ncausal_conv1d_update: FOUND")
except Exception as e:
    print(f"\ncausal_conv1d_update: NOT FOUND — {e}")

In [4]:
import mamba_ssm
from mamba_ssm.ops.triton.selective_state_update import selective_state_update
from causal_conv1d import causal_conv1d_fn, causal_conv1d_update


mamba_ssm.selective_state_update = selective_state_update # type: ignore

for name, val in [
    ("selective_state_update", mamba_ssm.selective_state_update), # type: ignore
    ("selective_scan_fn",      mamba_ssm.selective_scan_fn),
    ("mamba_inner_fn",         mamba_ssm.mamba_inner_fn),
    ("causal_conv1d_fn",       causal_conv1d_fn),
    ("causal_conv1d_update",   causal_conv1d_update),
]:
    print(f"{name:30s} → {bool(val):5}  {type(val)}")

all_present = all([
    mamba_ssm.selective_state_update, # type: ignore
    mamba_ssm.selective_scan_fn,
    mamba_ssm.mamba_inner_fn,
    causal_conv1d_fn,
    causal_conv1d_update,
])
print("Fast path available:", all_present)

## 1.3 Importy

In [5]:
import torch
import torch.nn as nn
import numpy as np
import evaluate
import datasets
from transformers import (
    AutoTokenizer, AutoConfig,
    MambaPreTrainedModel, MambaModel,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [6]:
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"] # type: ignore
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"] # type: ignore
    return {"accuracy": accuracy, "f1": f1}

# 2. Wczytanie datasetu

In [7]:
print("Is CUDA available?", torch.cuda.is_available())

In [8]:
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

In [9]:
print("\nExample from training set:")
print(dataset['train'][0])

# 3. Podejście 1: Pretrenowany transformer

Użycie Trainer z biblioteki `transformers` do fine-tuningu modelu — uczenie całego modelu bez mrożenia wag. 
Używamy modelu `distilbert-base-uncased` do klasyfikacji sentymentu.

## Setup DistilBERT

In [ ]:
import time
from transformers import (
    TrainerCallback,
    DistilBertForSequenceClassification,
    DistilBertConfig,
)

def fineTuneBert(
    max_length=512, batch_size=32, lr=2e-5, num_epochs=5, weight_decay=0.01, model_max_embed=None
):
    tokenizer = AutoTokenizer.from_pretrained(
        "distilbert/distilbert-base-uncased"
    )
    if model_max_embed is None:
        model_max_embed = max_length
        
    def tokenize_function(examples):
        return tokenizer(
            examples["text"], truncation=True, max_length=max_length
        ) 

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(
                f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds"
            )

    config = DistilBertConfig.from_pretrained(
        "distilbert/distilbert-base-uncased",
        max_position_embeddings=model_max_embed,
        num_labels=2,
        problem_type="single_label_classification"
    )
    
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert/distilbert-base-uncased",
        config=config,
        ignore_mismatched_sizes=True
    )
	eval_steps = max(1, len(tokenized_dataset["train"])//batch_size//10)
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="steps",
        save_strategy='steps',
        logging_strategy='steps',
        eval_steps=eval_steps,
        save_steps=eval_steps,
        logging_steps=eval_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=2,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        logging_dir="./logs",
        report_to="tensorboard", 
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            TimingCallback()
        ],  
    )

    results = trainer.evaluate()
    print("\nDistilBERT Initial Evaluation Results:")
    print(results)

    trainer.train()

    results = trainer.evaluate()
    print("\nDistilBERT Evaluation Results:")
    print(results)

In [11]:
import torch
from torch.utils.data import DataLoader

def inferenceTimEvalBERT(seq_len):
    print(f"\n--- Benchmarking Sequence Length: {seq_len} ---")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=seq_len)
        
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    test_set = tokenized_dataset["test"]
    
    config = DistilBertConfig(
        max_position_embeddings=512 if seq_len<=512 else 1024,
        num_labels=2, # type: ignore
        problem_type="single_label_classification",
    )
    model = DistilBertForSequenceClassification(config=config)
    model.to(device) # type: ignore
    model.eval()

    batch_size = 8
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    
    dataloader = DataLoader(
        test_set,  # type: ignore
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=data_collator
    )

    print("Warming up CUDA kernels...")
    warmup_batches = 3
    with torch.inference_mode():
        for i, batch in enumerate(dataloader):
            if i >= warmup_batches:
                break
            inputs = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            _ = model(input_ids=inputs, attention_mask=mask)

    if device == "cuda":
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record() # type: ignore
        
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                _ = model(input_ids=inputs, attention_mask=mask)
                
        end_event.record() # type: ignore
        torch.cuda.synchronize()
        total_time_ms = start_event.elapsed_time(end_event)
    else:
        start_time = time.perf_counter()
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                _ = model(input_ids=inputs, attention_mask=mask)
        total_time_ms = (time.perf_counter() - start_time) * 1000

    total_samples = len(test_set)
    avg_time_per_batch_ms = total_time_ms / len(dataloader)
    samples_per_second = total_samples / (total_time_ms / 1000)

    print(f"\n[Results for Seq Len {seq_len}]")
    print(f"Total Samples Processed: {total_samples}")
    print(f"Total GPU Wall Time:     {total_time_ms / 1000:.4f} seconds")
    print(f"Avg Time per Batch:      {avg_time_per_batch_ms:.2f} ms (Batch Size: {batch_size})")
    print(f"Throughput Speed:        {samples_per_second:.2f} samples/second")

## Sekwencje

### Długość sekwencji 128

In [52]:
fineTuneBert(128,batch_size=60,lr=2e-5,weight_decay=0.01,model_max_embed=512)

In [58]:
inferenceTimEvalBERT(128)

### Długość sekwencji 256

In [60]:
fineTuneBert(256,model_max_embed=512,batch_size=100)

In [12]:
inferenceTimEvalBERT(256)


### Długość sekwencji 512

In [ ]:
fineTuneBert(512,batch_size=16,model_max_embed=512)

In [13]:
inferenceTimEvalBERT(512)

### Długość sekwencji 1024

In [15]:
fineTuneBert(1024,batch_size=16,lr=1e-5,num_epochs=10,weight_decay=0.01,model_max_embed=1024)

In [14]:
inferenceTimEvalBERT(1024)

# 4. Podejście 2: Pretrenowana Mamba

Używamy modelu `state-spaces/mamba-130m`, który potem fine-tune'ujemy do zadania klasyfikacji. Biblioteka `transformers` nie oferuje gotowego modelu do tego zadania, więc napisaliśmy własny wrapper. 

## Setup mamby

In [ ]:
from transformers import GPTNeoXTokenizerFast # type: ignore

def prepare_mamba_dataset(dataset, max_length=1024):
    tokenizer = GPTNeoXTokenizerFast.from_pretrained("state-spaces/mamba-130m-hf")

    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
        )

    print("Tokenizing dataset for Mamba...")
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    if "label" in tokenized_dataset["train"].column_names:
        tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

    tokenized_dataset.set_format("torch")

    return tokenized_dataset, tokenizer

In [ ]:
import mamba_ssm

if not hasattr(mamba_ssm, 'selective_state_update'):
    from mamba_ssm.ops.selective_scan_interface import (
        selective_scan_fn,
        selective_state_update, # type: ignore
    )
    mamba_ssm.selective_state_update = selective_state_update # type: ignore
    mamba_ssm.selective_scan_fn     = selective_scan_fn

if not hasattr(mamba_ssm, 'mamba_inner_fn'):
    try:
        from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn
        mamba_ssm.mamba_inner_fn = mamba_inner_fn
    except ImportError:
        # v2 renamed / removed mamba_inner_fn; None makes transformers skip it
        mamba_ssm.mamba_inner_fn = None

print("mamba_ssm patched:",
      hasattr(mamba_ssm, 'selective_state_update'),
      hasattr(mamba_ssm, 'selective_scan_fn'),
      hasattr(mamba_ssm, 'mamba_inner_fn'))

In [18]:
class MambaForSequenceClassification(MambaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.backbone = MambaModel(config)  # must be 'backbone' to match checkpoint keys
        self.score = nn.Linear(config.hidden_size, self.num_labels, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, use_cache=False)
        hidden  = outputs[0]  # (B, L, H)

        if attention_mask is not None:
            seq_lens = attention_mask.int().sum(-1) - 1
            pooled   = hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]
        else:
            pooled   = hidden[:, -1, :]

        logits = self.score(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states)

In [ ]:
class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

def fineTuneMambaClassification(tokenized_dataset, collator,batch_size=16, lr=2e-5, num_epochs=5, weight_decay=0.01):
    tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
    tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True # Set this directly during config loading
    )

    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, # Pass the fully configured config
        ignore_mismatched_sizes=True
    )
    # Explicitly configure pad token ID to match the tokenizer configuration
    model.config.pad_token_id = tokenizer.pad_token_id

	eval_steps = max(1, len(tokenized_dataset["train"]//batch_size//10))
    training_args = TrainingArguments(
        output_dir="./mamba_classification_results",
        eval_strategy="steps",
        save_strategy='steps',
        logging_strategy='steps',
        eval_steps=eval_steps,
        save_steps=eval_steps,
        logging_steps=eval_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=2,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        gradient_checkpointing=False,
        weight_decay=weight_decay,
        fp16=False,  # Recommended for custom CUDA compilation layouts
        bf16=True,  # Recommended for custom CUDA compilation layouts
        report_to="tensorboard"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[TimingCallback()]
    )

    trainer.train()

In [ ]:
import torch

def inferenceTimEval(seq_len):
    print(f"\n--- Benchmarking Sequence Length: {seq_len} ---")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=seq_len)
    
    if mamba_tokenizer.pad_token is None:
        mamba_tokenizer.pad_token = mamba_tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True  
    )
    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, 
        ignore_mismatched_sizes=True
    ).to(device) # type: ignore
    model.eval()

    batch_size = 8
    
    test_set = tokenized_dataset["test"]
    
    data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer, return_tensors="pt")
    
    dataloader = DataLoader(
        test_set, # type: ignore
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=data_collator
    )

    print("Warming up CUDA kernels...")
    warmup_batches = 3
    with torch.inference_mode():
        for i, batch in enumerate(dataloader):
            if i >= warmup_batches:
                break
            inputs = batch["input_ids"].to(device)
            _ = model(inputs)

    if device == "cuda":
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record()# type: ignore
        
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                _ = model(inputs)
                
        end_event.record()# type: ignore
        torch.cuda.synchronize()
        total_time_ms = start_event.elapsed_time(end_event)
    else:
        start_time = time.perf_counter()
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                _ = model(inputs)
        total_time_ms = (time.perf_counter() - start_time) * 1000

    total_samples = len(test_set)
    avg_time_per_batch_ms = total_time_ms / len(dataloader)
    samples_per_second = total_samples / (total_time_ms / 1000)

    print(f"\n[Results for Seq Len {seq_len}]")
    print(f"Total Samples Processed: {total_samples}")
    print(f"Total GPU Wall Time:     {total_time_ms / 1000:.4f} seconds")
    print(f"Avg Time per Batch:      {avg_time_per_batch_ms:.2f} ms (Batch Size: {batch_size})")
    print(f"Throughput Speed:        {samples_per_second:.2f} samples/second")

## Długość sekwencji 128

In [ ]:
import sys
import torch # Import torch to get its __file__ attribute


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=128)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

In [26]:
inferenceTimEval(128)

## Długość sekwencji 256

In [ ]:
import sys
import torch # Import torch to get its __file__ attribute


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=256)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

In [28]:
inferenceTimEval(256)

## Długość sekwencji 512

In [26]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=512)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

In [29]:
inferenceTimEval(512)

## Długość sekwencji 1024

In [12]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=1024)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=8)

In [30]:
inferenceTimEval(1024)

# 5. Analiza skalowalności

Porównanie
-   **Czasu na epokę**
-   **Czasu inferencji**
-   **Końcowej jakości modelu**: Accuracy, F1


## 5.1 Jakość modeli

| Długość sekwencji | DistilBERT                | Mamba                      |
| :---------------- | :------------------------ | :------------------------- |
| 128               | ~87.6% Accuracy, 87.6% F1 | ~89.5% Accuracy, ~89.5% F1 |
| 256               | ~91.0% Accuracy, 91.0% F1 | ~92.7% Accuracy, ~92.7% F1 |
| 512               | ~93.1% Accuracy, 93.1% F1 | ~94.5% Accuracy, ~94.5% F1 |
| 1024              | ~88.0% Accuracy, 88.0% F1 | ~94.8% Accuracy, ~94.8% F1 |

## 5.2 Czas inferencji

| Model | Długość sekwencji | Całkowity czas na gpu (s) | Średni czas na Batch (ms) | Próbki na sekundę (samples/s) |
| :--- | :---: | :---: | :---: | :---: |
| **DistilBERT** | 128 | 22.9722 | 7.35 | 1088.27 |
| **DistilBERT** | 256 | 45.9677 | 14.71 | 543.86 |
| **DistilBERT** | 512 | 90.4429 | 28.94 | 276.42 |
| **DistilBERT** | 1024 | 140.0985 | 44.83 | 178.45 |
| **Mamba** | 128 | 69.7934 | 22.33 | 358.20 |
| **Mamba** | 256 | 130.9437 | 41.90 | 190.92 |
| **Mamba** | 512 | 247.5346 | 79.21 | 101.00 |
| **Mamba** | 1024 | 347.1559 | 111.09 | 72.01 |

## 5.3 Czas treningu

| Model          | Długość sekwencji | Epoch 1 (s) | Epoch 2 (s) | Epoch 3 (s) | Epoch 4 (s) | Epoch 5 (s) | Avg / Epoch  |
| :------------- | :---------------: | :---------: | :---------: | :---------: | :---------: | :---------: | :----------: |
| **DistilBERT** |        128        |    70.24    |    70.47    |    70.63    |    70.65    |    71.20    | **~70.64s**  |
| **DistilBERT** |        256        |   144.40    |   144.52    |   145.26    |   143.42    |   145.63    | **~144.65s** |
| **DistilBERT** |        512        |   317.19    |   320.88    |   320.39    |   311.89    |   312.27    | **~316.52s** |
| **DistilBERT** |       1024        |   588.33    |   587.76    |   583.76    |   584.24    |   589.98    | **~587.18s** |
| **Mamba**      |        128        |   169.40    |   167.07    |   167.20    |   165.89    |   166.93    | **~167.30s** |
| **Mamba**      |        256        |   253.29    |   254.39    |   253.70    |   253.79    |   255.26    | **~254.09s** |
| **Mamba**      |        512        |   445.90    |   448.73    |   450.96    |   444.77    |   446.99    | **~447.47s** |
| **Mamba**      |       1024        |   713.34    |   716.99    |   760.00    |   760.32    |   728.29    | **~735.79s** |

+ Dla obu architektur, przed fine-tuningiem, accuracy i f1 wynosi około 0.5 co jest spodziewane dla problemu klasyfikacji binarnej
+ Czas treningu dla `DistilBERT` zależy prawie liniowo od długości sekwencji, dla `mamby` czas ten rośnie nieliowo
+ `Mamba` trenuje się dłużej, ale skaluje się lepiej czasowo
+ Model `Mamba` jest około 2x większy niż `DistilBERT`(130M vs. 66M parametrów)
+ Dla modelu `mamba` zwiększenie długości sekwencji z 512 na 1024 przynosi niewielkie korzyści , dla modelu `DistilBERT` pogarsza wyniki
    + Jakość mamby, jako modelu rekurencyjnego, nie pogarsza się w tak jak BERT, ponieważ długości sekwencji, na której model był trenowany mogą być dowolne. Dla modelu BERT było to 512 tokenów